<div align="center" style="font-size:28px; font-weight:bold;">
MFE 409: Financial Risk Management
<br>Problem Set 5
<br>Valentin Haddad
<br>due 2/19 before midnight
</div>

<br>

Cohort 2, Group 6:
Lee James, Moazzami Ali, Yu Aiden, Cai Jenny, Li Zehao, Guo Lucy

## 1 VaR for option on two underlying

#### 1. We are interested in managing the risk of an option on two stocks with prices $S_{1,t}$ and $S_{2,t}$ . Assume a short rate r and:
$$
\begin{aligned}
\frac{dS_{1,t}}{S_{1,t}} &= \mu_1 dt + \sigma_1 dW_{1,t} \\
\frac{dS_{2,t}}{S_{2,t}} &= \mu_2 dt + \sigma_2 dW_{2,t} \\
corr(dW_{1,t},dW_{2,t}) &= \rho dt \\
\end{aligned}
$$

#### where all the parameters are in daily units. Call $M(S_{1,t}, S_{2,t})$ the price of the option. Derive a formula for the 99% 1-day VaR for the option using the delta approach.

#### Solution:

Apply first order Taylor expansion wrt each asset to approximate change in option's value, $\Delta M$.

$$
\Delta M \approx \frac{\partial M}{\partial S_1} \Delta S_1 + \frac{\partial M}{\partial S_2} \Delta S_2
$$

$$
\Delta M \approx \Delta_1 \Delta S_1 + \Delta_2 \Delta S_2
$$

where $\Delta_1 = \frac{\partial M}{\partial S_1}$ and $\Delta_2 = \frac{\partial M}{\partial S_2}$

We are given the stochastic differential equations for the stocks. Discretizing these over a 1-day horizon($\Delta t = 1$), we have:

$$\Delta S_1 = \mu_1 S_1 \Delta t + \sigma_1 S_1 \Delta W_1$$
$$\Delta S_2 = \mu_2 S_2 \Delta t + \sigma_2 S_2 \Delta W_2$$

Since $\Delta S_i = S_i * R_{i,t}$, where $R_{i,t} = \frac{\Delta S_i}{S_i}$ is the daily return of stock i, we can write:

$$
\Delta M \approx \Delta_1 S_1 R_{1,t}+ \Delta_2 S_2 R_{2,t}
$$

Since daily returns are normally distributed $(R_{1,t}\sim \mathcal{N}(\mu_{1},\sigma_{1}^{2})$, $R_{2,t}\sim \mathcal{N}(\mu_{2},\sigma_{2}^{2}))$, $\Delta M$ will be a linear combination of jointly normal random variables, thus normally distributed with $corr(R_{1,t}, R_{2,t}) = \rho$

So its variance is:
$$
\sigma_{\Delta M}^2 = (\Delta_1 S_1)^2\,\sigma_1^2 + (\Delta_2 S_2)^2\,\sigma_2^2 + 2\rho\,(\Delta_1 S_1)(\Delta_2 S_2)\,\sigma_1\sigma_2
$$

and its standard deviation is:
$$
\sigma_{\Delta M} = \sqrt{(\Delta_1 S_1)^2\,\sigma_1^2 + (\Delta_2 S_2)^2\,\sigma_2^2 + 2\rho\,(\Delta_1 S_1)(\Delta_2 S_2)\,\sigma_1\sigma_2}
$$

Thus 99% 1-day VaR is(using $z(0.99) = 2.326$):
$$
\begin{aligned}
VaR &= 2.326 * \sigma_{\Delta M} \\
VaR &= 2.326 * \sqrt{(\Delta_1 S_1)^2\,\sigma_1^2 + (\Delta_2 S_2)^2\,\sigma_2^2 + 2\rho\,(\Delta_1 S_1)(\Delta_2 S_2)\,\sigma_1\sigma_2}
\end{aligned}
$$

#### 2. Consider the case of a European option on the minimum of the two stock price, with maturity T and final payoff:
$$
\begin{aligned}
M_T &= max(min(S_{1,t},S_{2,t}) - K, 0)
\end{aligned}
$$

#### From Stulz (1982) (attached paper), the price of the option when the time to maturity is $\tau$ is:
$
\begin{aligned}
M(S_1,S_2) &= S_1 N_2 \left(\gamma_1 + \sigma_1\sqrt{\tau}, \frac{\ln(S_2/S_1)-\frac{1}{2}\sigma^2\sqrt{\tau}}{\sigma\sqrt{\tau}}, \frac{\rho\sigma_2-\sigma_1}{\sigma}\right) \\
&+ S_2 N_2 \left(\gamma_2 + \sigma_2\sqrt{\tau}, \frac{\ln(S_1/S_2)-\frac{1}{2}\sigma^2\sqrt{\tau}}{\sigma\sqrt{\tau}}, \frac{\rho\sigma_1-\sigma_2}{\sigma}\right) \\
&- Ke^{-r\tau}N_2(\gamma_1,\gamma_2,\rho)
\end{aligned}
$

where
$$
\begin{aligned}
\gamma_1 &= \frac{\ln(S_1/K) + (r - \frac{1}{2}\sigma_1^2)\tau}{\sigma_1 \sqrt{\tau}} \\
\gamma_2 &= \frac{\ln(S_2/K) + (r - \frac{1}{2}\sigma_2^2)\tau}{\sigma_2 \sqrt{\tau}} \\
\sigma^2 &= \sigma_1^2 + \sigma_2^2 - 2 \rho \sigma_1 \sigma_2
\end{aligned}
$$

#### and $N_2(\alpha, \beta, \theta)$ is the bivariate cumulative standard normal distribution with upper limits $\alpha$ and $\beta$ and correlation $\theta$. That is, if $X_1$ and $X_2$ are standard normal with correlation $\theta$:
$$
N_2(\alpha, \beta, \theta) = P(X_1 \leq \alpha, X_2 \leq \beta)
$$

#### Assume: $r = 0.02\%, \sigma_1 = \sigma_2 = 1.5\%, \rho = 0.35, \mu_1 = \mu_2 = 0.025\%$, where again all parameters are in daily units. Further assume that at date 0, we have T = 6 months, and that $S_{1,0} = 99$, $S_{2,0} = 101$ and $K = 100$. Compute the price of the option at date 0.

#### Solution:



In [13]:
import numpy as np
from scipy.stats import multivariate_normal

r = 0.0002
sigma1 = 0.015
sigma2 = 0.015
rho = 0.35
S1 = 99.0
S2 = 101.0
K = 100.0
tau = 126.0

sigma_sq = sigma1**2 + sigma2**2 - 2 * rho * sigma1 * sigma2
sigma = np.sqrt(sigma_sq)

gamma1 = (np.log(S1 / K) + (r - 0.5 * sigma1**2) * tau) / (sigma1 * np.sqrt(tau))
gamma2 = (np.log(S2 / K) + (r - 0.5 * sigma2**2) * tau) / (sigma2 * np.sqrt(tau))

def N2(alpha, beta, theta):
    """
    Computes the bivariate normal CDF N2(alpha, beta, theta)
    """
    cov_matrix = [[1.0, theta], [theta, 1.0]]
    return multivariate_normal.cdf([alpha, beta], mean=[0, 0], cov=cov_matrix)

# Term 1 Components
alpha1 = gamma1 + sigma1 * np.sqrt(tau)
beta1 = (np.log(S2 / S1) - 0.5 * sigma_sq * tau) / (sigma * np.sqrt(tau))
theta1 = (rho * sigma2 - sigma1) / sigma

term1 = S1 * N2(alpha1, beta1, theta1)

# Term 2 Components
alpha2 = gamma2 + sigma2 * np.sqrt(tau)
beta2 = (np.log(S1 / S2) - 0.5 * sigma_sq * tau) / (sigma * np.sqrt(tau))
theta2 = (rho * sigma1 - sigma2) / sigma

term2 = S2 * N2(alpha2, beta2, theta2)

# Term 3 Components
term3 = K * np.exp(-r * tau) * N2(gamma1, gamma2, rho)

M = term1 + term2 - term3

print(f"The price of the European option at date 0 is: {M:.4f}")

The price of the European option at date 0 is: 3.4173


#### 3. Compute the 99% 1-day VaR for the option using the formula you have derived in question 1.

In [14]:
delta1 = N2(alpha1, beta1, theta1)
delta2 = N2(alpha2, beta2, theta2)

var_S1_term = delta1 * S1 * sigma1
var_S2_term = delta2 * S2 * sigma2

variance_M = (var_S1_term**2) + (var_S2_term**2) + (2 * rho * var_S1_term * var_S2_term)
sigma_delta_M = np.sqrt(variance_M)

VaR_99 = 2.326 * sigma_delta_M

print(f"Delta 1 (w.r.t S1): {delta1:.4f}")
print(f"Delta 2 (w.r.t S2): {delta2:.4f}")
print(f"Standard Deviation of Option Change: {sigma_delta_M:.4f}")
print(f"99% 1-day VaR: {VaR_99:.4f}")

Delta 1 (w.r.t S1): 0.1916
Delta 2 (w.r.t S2): 0.1671
Standard Deviation of Option Change: 0.4421
99% 1-day VaR: 1.0282


#### 4. Compute the 99% 1-day VaR for the option using simulations. Compare to the results of the previous question and explain the intuition behind this result.

In [16]:
np.random.seed(1234)

mu1 = 0.00025
mu2 = 0.00025
N_sim = 100000

Z1 = np.random.standard_normal(N_sim)
Z2_indep = np.random.standard_normal(N_sim)
Z2 = rho * Z1 + np.sqrt(1 - rho**2) * Z2_indep

S1_sim = S1 * np.exp((mu1 - 0.5 * sigma1**2) * 1 + sigma1 * Z1)
S2_sim = S2 * np.exp((mu2 - 0.5 * sigma2**2) * 1 + sigma2 * Z2)

def stulz_price_vectorized(S1_arr, S2_arr, tau_val):
    gamma1_arr = (np.log(S1_arr / K) + (r - 0.5 * sigma1**2) * tau_val) / (sigma1 * np.sqrt(tau_val))
    gamma2_arr = (np.log(S2_arr / K) + (r - 0.5 * sigma2**2) * tau_val) / (sigma2 * np.sqrt(tau_val))

    alpha1_arr = gamma1_arr + sigma1 * np.sqrt(tau_val)
    beta1_arr = (np.log(S2_arr / S1_arr) - 0.5 * sigma_sq * tau_val) / (sigma * np.sqrt(tau_val))

    alpha2_arr = gamma2_arr + sigma2 * np.sqrt(tau_val)
    beta2_arr = (np.log(S1_arr / S2_arr) - 0.5 * sigma_sq * tau_val) / (sigma * np.sqrt(tau_val))

    cov1 = [[1.0, theta1], [theta1, 1.0]]
    cov2 = [[1.0, theta2], [theta2, 1.0]]
    cov3 = [[1.0, rho], [rho, 1.0]]

    N2_1_arr = multivariate_normal.cdf(np.column_stack((alpha1_arr, beta1_arr)), mean=[0, 0], cov=cov1)
    N2_2_arr = multivariate_normal.cdf(np.column_stack((alpha2_arr, beta2_arr)), mean=[0, 0], cov=cov2)
    N2_3_arr = multivariate_normal.cdf(np.column_stack((gamma1_arr, gamma2_arr)), mean=[0, 0], cov=cov3)

    term1_arr = S1_arr * N2_1_arr
    term2_arr = S2_arr * N2_2_arr
    term3_arr = K * np.exp(-r * tau_val) * N2_3_arr

    return term1_arr + term2_arr - term3_arr

M1_sim = stulz_price_vectorized(S1_sim, S2_sim, tau - 1.0)

losses = M - M1_sim

VaR_99_simulated = np.percentile(losses, 99)

print(f"99% 1-day VaR (Simulated): {VaR_99_simulated:.4f}")

99% 1-day VaR (Simulated): 0.9334


The simulated 99% 1-day VaR (0.9334) is noticeably lower than the Delta-Normal VaR (1.0282) derived in Question 3. The primary intuition behind this difference is that the Delta-Normal approach relies on a first-order, strictly linear approximation of the option's price. By only using Delta, it assumes the option's downside risk scales perfectly linearly with the stock prices, completely ignoring the second-order price sensitivity known as Gamma ($\Gamma$).

Because this is a long option position, it possesses positive Gamma (convexity). Positive Gamma means that as the underlying stock prices fall—the scenario that generates our VaR loss—the option's deltas decrease, naturally decelerating the rate of the loss. The linear Delta-Normal method misses this "cushioning" effect and therefore strictly overestimates the maximum expected loss. The Monte Carlo simulation, on the other hand, recalculates the exact, non-linear option price at the new simulated stock prices, fully capturing this beneficial convexity (along with the positive drift and time decay), resulting in a more accurate and lower VaR estimate.

#### 5. **Optional**: Derive a formula for the 99%-VaR for the option using the delta-gamma approach. Implement the formula and compare your results to the other two approaches. Explain the intuition for this result.

## SKIPPED

#### 6. Now assume you are a trader in the real world and you do not know for sure that the model for the underlying is correct. What other types of risks would you worry about? If you had to worry about just one more risk, what would it be? Explain (quantitatively) how you get to this conclusion.

#### Solution:

If the geometric Brownian motion (GBM) model is inaccurate, a trader faces volatility, liquidity, jump, and correlation risks. For an option based on the minimum of two stocks, assuming a constant correlation ($\rho$) is highly dangerous. In reality, asset correlations often spike toward 1.0 during market crashes, causing both underlying stocks to plummet simultaneously and severely reducing the option's expected payoff.

If forced to isolate just one major risk, it would be Tail Risk (Excess Kurtosis). The GBM model assumes perfectly normal log-returns, which is why we used a standard multiplier of 2.326 for the 99% VaR. However, actual equity returns have "fat tails" and negative skewness.  Quantitatively, if real-world returns follow a heavy-tailed distribution, the true 99th percentile drop could easily represent 3 or 4 standard deviations. This implies that the standard normal model significantly underestimates the true Value at Risk, leaving the trader undercapitalized for extreme market downturns.

## 2 Case Study: Implementing Quantitative Risk Management and VaR in a Chinese Investment Bank

### You should buy the case study material using this link: https://hbsp.harvard.edu/import/1391580.

# NOTE: Group Work Submitted by Zehao Li

#### 1. Explain the objectives and priorities of each player: Jasper Wang, Jianguo Lu, and Charles Pan. What is motivating the different players? What tensions existed among their different objectives?

#### 2. Why does Jasper choose to make the VaR model the first step towards rationalizing the trading function? What is the appeal of the VaR model generally?

#### 3. Why do Jianguo and the traders resist the VaR model? Do you think their pattern of resistance to risk management is unique to China, or might it be found elsewhere too?

#### 4. Using the spreadsheet provided, run backtests of the VaR predictions against actual daily gains or losses for both the S&P 500 index and the Shanghai Composite index.

#### (a) Starting with a lookback period of three months, observe the number of exceptions in all years for both the Shanghai and S&P indexes. How do they compare?

#### (b) Try different lookback periods (say, 3, 6 and 9 months) to see if the length of the period changes your conclusions.

#### (c) Given that Jasper's VaR model assumes a 95% confidence level, how well does the backtest validate the model?

#### 5. How might Jasper use the backtest results to bolster his case for introducing the VaR model?

#### 6. How successful do you think Jasper will be in his attempt to implement Western risk management practices? What advice would you give to someone in a role similar to his?

#### 7. What is the current regulation environment of risk followed by Chinese banks and how has it evolved since the crisis? (Find information beyond the case study material)